## Extracción de datos de empresas disueltas desde la API del INE

### Objetivo
Este notebook extrae y estructura los datos de **sociedades mercantiles disueltas** en España a través de la API pública TEMPUS del INE (tabla 13915).

Los datos se desglosan por **territorio**, **causa de disolución** (Voluntaria, Por fusión, Otras) y **periodo mensual**, obteniendo el número de sociedades disueltas en cada segmento.

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/13915`.
2. **Extracción y filtrado** — Recorrido de la respuesta JSON excluyendo registros nacionales y agrupaciones "Total" para evitar duplicados.
3. **Estructuración** — Poblado de un diccionario con las columnas: `id_dis`, `territorio`, `id_tiempo`, `razon` y `numero_sociedades`.
4. **Exportación** — Volcado a CSV en `../files/data_raw/empresas_disueltas.csv`.

### Contexto del proyecto
Estos datos se integran en un análisis de **resiliencia empresarial en España**, donde se cruzarán con constituciones de empresas e IPC para estudiar la relación entre el entorno macroeconómico y la mortalidad empresarial.

In [1]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd

# Esto obliga a Python a mirar hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))

# Importación del módulo de conexión a la API
from src.api import connection_api
from src.api.config import API_URLS

c:\Users\fabih\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Paso 1: Conexión a la API del INE

In [2]:
url_dis = API_URLS["disueltas"]    #Disueltas

In [3]:
data_dis = connection_api.llamada_api(url_dis)

## Paso 2: Exploración de la estructura de datos

In [4]:
len(data_dis)

80

In [5]:
for serie in data_dis[:5]:
    print(serie['Nombre'])

Total. Disueltas. Número de Sociedades. Mercantiles. Total Nacional. 
Andalucía. Total. Disueltas. Número de Sociedades. Mercantiles. 
Aragón. Total. Disueltas. Número de Sociedades. Mercantiles. 
Asturias, Principado de. Total. Disueltas. Número de Sociedades. Mercantiles. 
Balears, Illes. Total. Disueltas. Número de Sociedades. Mercantiles. 


## Paso 3: Procesamiento y transformación

In [6]:
empresas_disueltas = { 
    'id_dis': [], 
    'territorio': [], 
    'id_tiempo': [],
    'razon': [], 
    'numero_sociedades': []            
}
contador = 1

for serie in data_dis:
    nombre_completo = serie['Nombre']
    
    if "nacional" not in nombre_completo.lower():
        
        partes = nombre_completo.split('.')
        territorio = partes[0].strip()
        razon = partes[1].strip()

        # P
        # rocesa y agrega si la razón NO es "Total"
        if razon.lower() != "total":

            for data in serie['Data']:
                id_tiempo = str(data['Anyo']) + str(data['FK_Periodo']).zfill(2)

                empresas_disueltas['id_dis'].append(contador)
                empresas_disueltas['territorio'].append(territorio)
                empresas_disueltas['id_tiempo'].append(id_tiempo)
                empresas_disueltas['razon'].append(razon)
                empresas_disueltas['numero_sociedades'].append(int(data['Valor']))
                            
                contador += 1

## Paso 4: Verificación del resultado

In [7]:
print(f"Filas: {len(empresas_disueltas['id_dis'])}")
print(f"Territorios: {set(empresas_disueltas['territorio'])}")
print(f"Rango temporal: {min(empresas_disueltas['id_tiempo'])} → {max(empresas_disueltas['id_tiempo'])}")

Filas: 12540
Territorios: {'Canarias', 'Comunitat Valenciana', 'País Vasco', 'Castilla - La Mancha', 'Madrid, Comunidad de', 'Castilla y León', 'Balears, Illes', 'Ceuta', 'Aragón', 'Melilla', 'Andalucía', 'Murcia, Región de', 'Rioja, La', 'Cantabria', 'Extremadura', 'Asturias, Principado de', 'Galicia', 'Navarra, Comunidad Foral de', 'Cataluña'}
Rango temporal: 200801 → 202604


## Paso 5: Exportación a CSV

In [8]:
pd.DataFrame(empresas_disueltas).to_csv('../files/data_raw/empresas_disueltas.csv', index=False)